In [ ]:
!pip install -q kailash kailash-ml polars torch torchvision matplotlib python-dotenv torch-geometric


# MLFP05 — Exercise 6.3: GraphSAGE (Sample and Aggregate)


**WHAT YOU'LL LEARN:**
- Why GCN doesn't scale to large graphs (full adjacency in memory)
- Inductive learning: generalise to unseen nodes at inference time
- Neighbour sampling strategy: fixed-size random subsets per node
- Separate self/neighbour projections for richer representations
- Train a scalable node classifier on the Cora citation network
- Track training with kailash-ml ExperimentTracker


**PREREQUISITES: M5/ex_6.1 (GCN), M5/ex_6.2 (GAT).**

**ESTIMATED TIME: ~30 min**


In [ ]:
from __future__ import annotations

import asyncio

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from shared.mlfp05.ex_6 import (
    OUTPUT_DIR,
    device,
    load_graph_data,
    plot_graph_with_embeddings,
    plot_node_embeddings,
    plot_training_curves,
    register_model,
    setup_engines,
    train_node_classifier,
)
from kailash_ml.types import MetricSpec

import matplotlib.pyplot as plt


## PHASE 1 — THEORY: Why GCN Doesn't Scale
GCN and GAT both require the FULL adjacency matrix during training.
For Cora (2,708 nodes), that's a 2708 x 2708 matrix — no problem.
But real-world graphs are much bigger:
- Singapore food delivery network: ~500K users x ~50K restaurants
- Facebook social graph: 3 billion nodes
- Google Knowledge Graph: 500 billion edges
A 500K x 500K dense adjacency matrix needs ~1 TB of memory. Even
sparse representations strain GPU memory when you need multi-hop
neighbourhood aggregation.
GraphSAGE (SAmple and aggrEGATE) solves this with three key ideas:
1. SAMPLE: Instead of aggregating ALL neighbours, randomly sample
a fixed number K (e.g., 10) per node. This bounds memory usage
regardless of graph size.
2. AGGREGATE: Use a learnable aggregator (mean, LSTM, or pooling)
over the sampled neighbours. The aggregator is a FUNCTION, not
a lookup table — it works on any set of neighbours.
3. INDUCTIVE: Because GraphSAGE learns an aggregation FUNCTION
rather than per-node embeddings, it can generalise to nodes it
has never seen during training. A new restaurant added to the
platform can be classified immediately using its neighbours.
The formula:
h'_i = sigma( W_self @ h_i + W_neigh @ MEAN(h_j for j in Sample(N(i))) )
Notice: SEPARATE weight matrices for self (W_self) and neighbours
(W_neigh). This lets the model learn different transformations for
"what I know about myself" vs "what my neighbours tell me".


In [ ]:
print("=" * 70)
print("  PHASE 1 — THEORY: GraphSAGE — Sample and Aggregate")
print("=" * 70)
print(
    """
  WHY GCN DOESN'T SCALE:
  - GCN needs full adjacency matrix in memory: O(N^2) for dense, O(E) for sparse
  - Cora (2.7K nodes): fine. Real graphs (500K+ nodes): memory explosion
  - Multi-hop aggregation expands exponentially: 2 hops of degree-50 = 2,500 nodes

  GRAPHSAGE'S THREE KEY IDEAS:

  1. SAMPLE: randomly pick K neighbours per node (not ALL)
     -> Fixed memory budget regardless of graph size
     -> Like dropout: different samples each epoch = regularisation

  2. AGGREGATE: learnable function over sampled neighbours
     -> Mean, LSTM, or pooling aggregator
     -> A FUNCTION, not a lookup — works on any neighbour set

  3. INDUCTIVE: learns HOW to aggregate, not WHAT to embed
     -> New nodes at inference time? No problem — just sample their
        neighbours and run the learned aggregator
     -> GCN/GAT are TRANSDUCTIVE: they need the full graph at test time

  Formula: h'_i = sigma( W_self @ h_i + W_neigh @ MEAN(sample(N(i))) )
  Separate W_self and W_neigh = "what I know" vs "what neighbours say"
"""
)


## PHASE 2 — BUILD: GraphSAGE Layer Implementation


In [ ]:
print("=" * 70)
print("  PHASE 2 — BUILD: GraphSAGE Layer + Model")
print("=" * 70)

# Load graph data and set up engines
graph_data = load_graph_data()
conn, tracker, exp_name, registry, has_registry = setup_engines()

X = graph_data["X"]
A = graph_data["A"]
y = graph_data["y"]
y_np = graph_data["y_np"]
A_np = graph_data["A_np"]
N = graph_data["N"]
F_dim = graph_data["F_dim"]
n_classes = graph_data["n_classes"]
dataset_name = graph_data["dataset_name"]

HIDDEN_DIM = 16 if dataset_name == "Karate Club" else 64
EPOCHS = 100
SAMPLE_K = 10  # Max neighbours to sample per node


class GraphSAGELayer(nn.Module):
    """Single GraphSAGE layer with mean aggregator and neighbour sampling.

    During training: randomly sample at most K neighbours per node
    (regularisation + scalability). At eval: use all neighbours for
    deterministic output (like dropout).

    Separate W_self and W_neigh allow the model to learn different
    transformations for a node's own features vs its neighbours'.
    """

    def __init__(self, in_dim: int, out_dim: int, sample_k: int = 10):
        super().__init__()
        # TODO: Create two linear projections (no bias):
        # - self.W_self: in_dim -> out_dim (for the node's own features)
        # - self.W_neigh: in_dim -> out_dim (for the aggregated neighbour features)
        # Also store self.sample_k = sample_k
        # Hint: nn.Linear(in_dim, out_dim, bias=False)
        pass

    def forward(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        n = h.size(0)

        # TODO: Implement neighbour sampling
        # During training (self.training == True) and sample_k < n:
        #   For each node i, find its neighbours, keep at most sample_k
        #   Build a sample_mask tensor of shape (n, n) with 1s for kept neighbours
        # During eval: use the full adjacency matrix
        # Hint: Loop over nodes, use torch.where(adj[i] > 0)[0] to find neighbours,
        #        then torch.randperm(len(neigh_idx))[:self.sample_k] to subsample
        if self.training and self.sample_k < n:
            # TODO: Build sample_mask by iterating over nodes
            # sample_mask = torch.zeros_like(adj)
            # for i in range(n):
            #     neigh_idx = torch.where(adj[i] > 0)[0]
            #     if len(neigh_idx) <= self.sample_k:
            #         sample_mask[i, neigh_idx] = 1.0
            #     else:
            #         perm = torch.randperm(len(neigh_idx), device=h.device)[:self.sample_k]
            #         sample_mask[i, neigh_idx[perm]] = 1.0
            # adj_sampled = sample_mask
            adj_sampled = adj  # Replace with your sampling implementation
        else:
            adj_sampled = adj

        # TODO: Mean aggregation + combine self and neighbour representations
        # 1. Compute degree: deg_sampled = adj_sampled.sum(dim=1, keepdim=True).clamp(min=1.0)
        # 2. Mean of neighbours: h_neigh = (adj_sampled @ h) / deg_sampled
        # 3. Self projection: h_self = self.W_self(h)
        # 4. Neighbour projection: h_agg = self.W_neigh(h_neigh)
        # 5. Combine: return h_self + h_agg
        pass


class GraphSAGE(nn.Module):
    """Two-layer GraphSAGE for node classification."""

    def __init__(
        self, in_dim: int, hidden_dim: int, n_classes: int, sample_k: int = 10
    ):
        super().__init__()
        # TODO: Create two GraphSAGE layers with sample_k
        # Layer 1: in_dim -> hidden_dim
        # Layer 2: hidden_dim -> n_classes
        pass

    def forward(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        # TODO: Two-layer forward: ReLU after layer 1, dropout(0.5), then layer 2
        pass

    def embed(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        """Return the hidden-layer embedding (before classification head)."""
        # TODO: Return ReLU(layer1(h, adj))
        pass


sage = GraphSAGE(
    in_dim=F_dim, hidden_dim=HIDDEN_DIM, n_classes=n_classes, sample_k=SAMPLE_K
)
n_params = sum(p.numel() for p in sage.parameters())
print(f"\n  GraphSAGE architecture:")
print(f"    Layer 1: GraphSAGELayer({F_dim} -> {HIDDEN_DIM}, sample_k={SAMPLE_K})")
print(f"    Layer 2: GraphSAGELayer({HIDDEN_DIM} -> {n_classes}, sample_k={SAMPLE_K})")
print(f"    Total parameters: {n_params:,}")
print(f"\n  How sampling works:")
print(f"    Training: each node samples up to {SAMPLE_K} neighbours per epoch")
print(f"    Eval: use all neighbours (deterministic, like dropout)")
print(f"    Effect: regularisation + bounded memory per mini-batch")

# Show how sampling affects neighbourhood size
degrees = A_np.sum(axis=1)
nodes_needing_sample = (degrees > SAMPLE_K).sum()
print(f"\n  Sampling impact on {dataset_name}:")
print(f"    Avg degree: {degrees.mean():.1f}")
print(f"    Max degree: {int(degrees.max())}")
print(f"    Nodes with degree > {SAMPLE_K}: {nodes_needing_sample} / {N}")
print(f"    -> {nodes_needing_sample} nodes will have sampled neighbourhoods")


### Build Checkpoint ────────────────────────────────────────────────


In [ ]:
assert isinstance(sage, nn.Module), "GraphSAGE should be an nn.Module"
assert n_params > 0, "GraphSAGE should have learnable parameters"
print("\n--- Build checkpoint passed --- GraphSAGE architecture created\n")


## PHASE 3 — TRAIN: Node Classification on Cora


In [ ]:
print("=" * 70)
print(f"  PHASE 3 — TRAIN: GraphSAGE on {dataset_name}")
print("=" * 70)

sage_losses, sage_val, sage_test = train_node_classifier(
    model=sage,
    name="GraphSAGE",
    forward_arg=A,
    graph_data=graph_data,
    tracker=tracker,
    exp_name=exp_name,
    epochs=EPOCHS,
)


### Train Checkpoint ────────────────────────────────────────────────


In [ ]:
assert len(sage_losses) == EPOCHS, f"Expected {EPOCHS} epoch losses for GraphSAGE"
assert sage_losses[-1] < sage_losses[0], "GraphSAGE loss should decrease"
best_val = max(sage_val)
best_test = max(sage_test)
print(f"\n  GraphSAGE Results:")
print(f"    Best validation accuracy: {best_val:.4f}")
print(f"    Best test accuracy:       {best_test:.4f}")
print(f"    Final loss:               {sage_losses[-1]:.4f}")
# INTERPRETATION: GraphSAGE is INDUCTIVE — it learns a generalised
# aggregation function that works on unseen nodes. During training, it
# randomly samples K neighbours per node (like dropout for graphs),
# which provides regularisation and makes it scalable to large graphs.
# The separate W_self and W_neigh projections let the model learn
# different transformations for a node's own features versus its
# neighbours' features.
print("\n--- Train checkpoint passed --- GraphSAGE trained successfully\n")


## PHASE 4 — VISUALISE: Embeddings + Sampling Effect


In [ ]:
print("=" * 70)
print("  PHASE 4 — VISUALISE: GraphSAGE Embeddings + Sampling Analysis")
print("=" * 70)

sage.eval()
with torch.no_grad():
    sage_emb = sage.embed(X, A).cpu().numpy()

# Plot 1: 2-D PCA of node embeddings
coords = plot_node_embeddings(
    embeddings=sage_emb,
    labels=y_np,
    n_classes=n_classes,
    title=f"GraphSAGE Node Embeddings — {dataset_name}",
    filename="sage_node_embeddings.png",
)

# Plot 2: Graph structure on embedding space
plot_graph_with_embeddings(
    A_np=A_np,
    embeddings_2d=coords,
    labels=y_np,
    n_classes=n_classes,
    title=f"GraphSAGE — Graph Structure in Embedding Space ({dataset_name})",
    filename="sage_graph_embeddings.png",
)

# Plot 3: Training curves
plot_training_curves(
    metrics_dict={"GraphSAGE train loss": sage_losses},
    title="GraphSAGE Training Loss",
    y_label="Cross-Entropy Loss",
    filename="sage_loss_curve.html",
)
plot_training_curves(
    metrics_dict={
        "GraphSAGE val accuracy": sage_val,
        "GraphSAGE test accuracy": sage_test,
    },
    title="GraphSAGE Accuracy",
    y_label="Accuracy",
    filename="sage_accuracy_curves.html",
)

# TODO: Analyse the effect of sampling K on embedding variance
# Run multiple forward passes in training mode to show stochastic embeddings
# 1. Set sage.train() to enable sampling
# 2. Run 5 forward passes with torch.no_grad(), collecting embeddings
# 3. Stack embeddings: shape (5, N, hidden)
# 4. Compute per-node variance: var across the 5 trials, mean across hidden dim
# 5. Set sage.eval() to restore deterministic mode
# 6. Create 2 subplots:
#    Left: histogram of per-node embedding variance
#    Right: scatter plot of variance vs node degree
# 7. Save to OUTPUT_DIR / "sage_sampling_variance.png"
# Hint: embeddings_stack = np.stack(embeddings_list)
#        per_node_var = embeddings_stack.var(axis=0).mean(axis=1)
print("\n  Sampling stochasticity analysis:")
embedding_variances = []
sage.train()  # Enable sampling
with torch.no_grad():
    embeddings_list = []
    for trial in range(5):
        emb_trial = sage.embed(X, A).cpu().numpy()
        embeddings_list.append(emb_trial)
    embeddings_stack = np.stack(embeddings_list)  # (5, N, hidden)
    per_node_var = embeddings_stack.var(axis=0).mean(axis=1)  # (N,)

sage.eval()  # Restore eval mode

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TODO: Left subplot — histogram of per_node_var
# axes[0].hist(per_node_var, bins=50, color="coral", edgecolor="white", alpha=0.8)
# axes[0].set_xlabel("Embedding Variance Across Samples")
# axes[0].set_title("Stochastic Embedding Variance\n(5 forward passes with sampling)")

# TODO: Right subplot — scatter of degrees vs per_node_var
# axes[1].scatter(degrees, per_node_var, s=8, alpha=0.4, color="steelblue")
# axes[1].set_xlabel("Node Degree")
# axes[1].set_ylabel("Embedding Variance")
# axes[1].set_title(f"Variance vs Degree (sample_k={SAMPLE_K})")

plt.tight_layout()
filepath = OUTPUT_DIR / "sage_sampling_variance.png"
plt.savefig(filepath, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  Saved: {filepath}")

high_var_nodes = (per_node_var > np.percentile(per_node_var, 90)).sum()
low_var_nodes = (per_node_var < np.percentile(per_node_var, 10)).sum()
print(f"    High-variance nodes (top 10%): {high_var_nodes} — mostly high-degree nodes")
print(f"    Low-variance nodes (bottom 10%): {low_var_nodes} — mostly low-degree nodes")
print(f"    Nodes with degree <= {SAMPLE_K}: deterministic (no sampling needed)")


### Visualise Checkpoint ────────────────────────────────────────────


In [ ]:
assert sage_emb.shape == (
    N,
    HIDDEN_DIM,
), f"Embedding shape should be ({N}, {HIDDEN_DIM})"
print("\n--- Visualise checkpoint passed --- GraphSAGE embeddings + variance plotted\n")


## PHASE 5 — APPLY: Recommendation Engine for Food Delivery


In [ ]:
print("=" * 70)
print("  PHASE 5 — APPLY: Food Delivery Recommendations (GrabFood/foodpanda)")
print("=" * 70)
print(
    """
  SCENARIO: You're building a recommendation engine for a Singapore food
  delivery platform (GrabFood, foodpanda, or Deliveroo).

  THE GRAPH:
  - User nodes: ~500K users with features (location, order frequency, cuisine prefs)
  - Restaurant nodes: ~50K restaurants (cuisine type, price range, rating)
  - Edges: user-restaurant orders (weighted by frequency)
  - Bipartite graph: users only connect to restaurants, not to each other

  WHY GRAPHSAGE IS THE RIGHT CHOICE:
  1. SCALE: 550K nodes = GCN's adjacency matrix would need 302 billion entries
     GraphSAGE samples 10 neighbours per node = bounded memory
  2. INDUCTIVE: new restaurants join daily. GCN would need to retrain on the
     entire graph. GraphSAGE classifies new restaurants immediately by
     sampling their first customers' embeddings.
  3. COLD START: a new restaurant with just 3 orders can be embedded —
     GraphSAGE averages those 3 users' embeddings. GCN has no mechanism
     for unseen nodes.

  RECOMMENDATION PIPELINE:
  1. Train GraphSAGE on the user-restaurant graph
  2. Each user gets an embedding (captures taste preferences via neighbours)
  3. Each restaurant gets an embedding (captures customer base profile)
  4. Score = dot_product(user_embedding, restaurant_embedding)
  5. Rank restaurants by score for each user -> personalised recommendations
"""
)

# TODO: Demonstrate collaborative filtering baseline vs GraphSAGE
# Using Cora as proxy: predict class from majority class of neighbours
# 1. For each node, find neighbours in A
# 2. Take majority vote of neighbour labels -> prediction
# 3. Compute accuracy on test_mask
# 4. Compare to GraphSAGE test accuracy
# Hint: counts = torch.bincount(neighbour_labels, minlength=n_classes)
#        majority_preds[i] = counts.argmax()
print("  Collaborative Filtering Baseline vs GraphSAGE:")

majority_preds = torch.zeros(N, dtype=torch.long, device=device)
test_mask = graph_data["test_mask"]

# TODO: Implement majority-vote baseline
# for i in range(N):
#     neighbours = torch.where(A[i] > 0)[0]
#     if len(neighbours) == 0:
#         majority_preds[i] = 0
#         continue
#     neighbour_labels = y[neighbours]
#     counts = torch.bincount(neighbour_labels, minlength=n_classes)
#     majority_preds[i] = counts.argmax()

cf_acc = (majority_preds[test_mask] == y[test_mask]).float().mean().item()

print(f"    Collaborative filtering (neighbour majority): {cf_acc:.4f}")
print(f"    GraphSAGE (learned aggregation):              {best_test:.4f}")
improvement = best_test - cf_acc
print(
    f"    Improvement:                                  +{improvement:.4f} ({improvement*100:.1f} pp)"
)

print(
    """
  WHY GRAPHSAGE BEATS SIMPLE COLLABORATIVE FILTERING:
  - CF just counts neighbours — GraphSAGE LEARNS what to aggregate
  - CF has no features — GraphSAGE combines structure with node features
  - CF is one-hop — 2-layer GraphSAGE captures 2-hop patterns
  - CF is fixed — GraphSAGE adapts its aggregation during training

  DEPLOYMENT CONSIDERATIONS:
  1. Mini-batch training: sample subgraphs, not full graph (torch_geometric)
  2. Pre-compute embeddings offline; serve recommendations from cache
  3. Retrain weekly with new orders; update embeddings for active users daily
  4. Track recommendation quality with ExperimentTracker (CTR, order rate)
  5. A/B test: GraphSAGE recs vs popularity-based recs vs CF baseline
"""
)

# Register the GraphSAGE model
if has_registry:
    version = register_model(
        registry=registry,
        name=f"m5_graphsage_{dataset_name.lower().replace(' ', '_')}",
        model=sage,
        metrics=[
            MetricSpec(name="best_val_accuracy", value=best_val),
            MetricSpec(name="best_test_accuracy", value=best_test),
            MetricSpec(name="final_loss", value=sage_losses[-1]),
            MetricSpec(name="cf_baseline_accuracy", value=cf_acc),
            MetricSpec(name="improvement_over_cf", value=improvement),
            MetricSpec(name="sample_k", value=float(SAMPLE_K)),
            MetricSpec(name="hidden_dim", value=float(HIDDEN_DIM)),
            MetricSpec(name="epochs", value=float(EPOCHS)),
        ],
    )
    print(f"  Registered GraphSAGE: version={version.version}, val_acc={best_val:.4f}")


### Apply Checkpoint ────────────────────────────────────────────────


In [ ]:
assert cf_acc > 0.0, "CF baseline should produce non-zero accuracy"
print("\n--- Apply checkpoint passed --- recommendation scenario demonstrated\n")


## REFLECTION


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — GraphSAGE")
print("=" * 70)
print(
    f"""
  GRAPHSAGE (Hamilton, Ying & Leskovec, 2017):
  [x] Neighbour sampling: fixed K neighbours per node bounds memory
  [x] Mean aggregator: MEAN(h_j for j in Sample(N(i)))
  [x] Separate projections: W_self @ h_i + W_neigh @ h_agg
  [x] INDUCTIVE learning: generalises to unseen nodes (new restaurants!)
  [x] Trained on {dataset_name}: {best_val:.1%} val accuracy, {best_test:.1%} test accuracy
  [x] Analysed sampling stochasticity: high-degree nodes -> more variance
  [x] Beat collaborative filtering baseline by {improvement*100:.1f} percentage points

  THREE-WAY COMPARISON (so far):
  - GCN: fixed weights, full graph, fast, simple
  - GAT: learned attention, full graph, interpretable
  - GraphSAGE: sampling + learned aggregation, SCALABLE, INDUCTIVE

  WHEN TO USE GRAPHSAGE:
  - Graph has 100K+ nodes (GCN/GAT memory-bound)
  - New nodes arrive at inference time (inductive requirement)
  - Mini-batch training needed (can't fit full graph on GPU)
  - Recommendation systems, dynamic social networks, evolving knowledge graphs

  Next: Exercise 6.4 — Link Prediction: predict missing edges with a
  dot-product decoder (foundation of recommendation systems)...
"""
)

# Clean up
await conn.close()
